# الدرس 12 - تقليل سجل الدردشة باستخدام مساحة عمل الوكيل

يعرض هذا الدفتر كيفية إدارة السياق في المحادثات الطويلة باستخدام إطار عمل الوكيل من مايكروسوفت. مع ازدياد المحادثات، يزداد عدد الرموز - مما يتجاوز في نهاية المطاف نافذة السياق للنموذج. نتعامل مع هذا باستخدام **نمطة تلخيص السياق** و **مساحة عمل الوكيل** للذاكرة الدائمة.

## ما ستتعلمه:
1. **لماذا إدارة السياق مهمة**: فهم حدود الرموز ونوافذ السياق
2. **الوكلاء الواعيون بالسياق**: بناء وكلاء يديرون سياق محادثاتهم بأنفسهم
3. **نمطة تلخيص السياق**: استخدام الأدوات لتكثيف تاريخ المحادثة
4. **مساحة عمل الوكيل**: ذاكرة دائمة تبقى بعد تقليل السياق

## المتطلبات المسبقة:
- إعداد Azure OpenAI مع تكوين متغيرات البيئة
- فهم مفاهيم الوكيل الأساسية من الدروس السابقة


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## لماذا إدارة السياق مهمة

لدى كل نموذج لغوي كبير (LLM) **نافذة سياق** محدودة — وهي الحد الأقصى لعدد الرموز التي يمكنه معالجتها في طلب واحد. مع تقدم المحادثة متعددة الأدوار:

- **عدد الرموز ينمو خطيًا** مع كل رسالة من المستخدم ورد من المساعد.
- **رموز الموجه تهيمن على التكلفة** لأن التاريخ الكامل يعاد إرساله في كل دور.
- في النهاية تتجاوز المحادثة **نافذة السياق** والنموذج إما يقتطع النص أو يخطئ.

### استراتيجيات إدارة السياق

| الاستراتيجية | كيف تعمل | المقايضة |
|---|---|---|
| **الاقتطاع** | حذف أقدم الرسائل | يفقد السياق المبكر |
| **التلخيص** | تكثيف الرسائل القديمة في ملخص | يتم فقدان بعض التفاصيل، لكن النقاط الرئيسية تبقى |
| **لوح مسح / ذاكرة خارجية** | تخزين الحقائق الرئيسية خارج المحادثة | يتطلب استدعاء أدوات، لكنه يصمد أمام أي تقليل |

في هذه الدفترية نجمع بين **التلخيص** و**أداة اللوح المسح** حتى يتمكن الوكيل من الحفاظ على الاستمرارية حتى عندما يتم تكثيف تاريخ المحادثة.


## إنشاء وكيل مدرك للسياق


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## محاكاة محادثة طويلة

دعونا نمر بمحادثة متعددة الأدوار لنرى كيف يتراكم السياق. يجب أن يحتفظ الوكيل بالتفاصيل الأساسية (التفضيلات، الميزانية، تواريخ السفر) عبر الأدوار وأن يبرهن على الاستمرارية.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

لاحظ كيف يحتفظ الوكيل بالسياق من الجولات السابقة — فهو يعرف عن اليابان، السوشي، المعابد، التصوير الفوتوغرافي، ميزانية 3000 دولار، السفر الفردي، وفترة أبريل. في حديث قصير هذا يعمل بشكل جيد، لكن مع ازدياد الحديث يصبح من المكلف إعادة إرسال التاريخ الكامل.

لنواصل المحادثة مع المزيد من الجولات لنرى تراكم السياق:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## نمط تلخيص السياق

مع نمو المحادثة، يمكننا استخدام **أداة التلخيص** لتكثيف السياق المتراكم في شكل مضغوط. يقوم الوكيل باستدعاء هذه الأداة لتسجيل التفضيلات الرئيسية بحيث يتم الاحتفاظ بالمعلومات الأساسية حتى إذا تم حذف الرسائل الأقدم.

هذا النمط هو عنصر البناء لتقليل التاريخ بشكل أكثر تقدمًا:
1. يحدد الوكيل الحقائق الرئيسية من المحادثة
2. يستدعي أداة التلخيص لحفظها
3. يمكن إزالة الرسائل الأقدم بأمان لأن الملخص يلتقط ما يهم

أدناه نعرف أداة `summarize_preferences` التي يمكن للوكيل استدعاؤها لتسجيل ملخص مضغوط لما تعلمه.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## ملخص

في هذا الدرس تعلمت كيفية إدارة السياق في محادثات الوكلاء طويلة الأمد باستخدام إطار عمل وكيل مايكروسوفت:

### المفاهيم الرئيسية
- **نوافذ السياق محدودة** — كل رمز مميز في تاريخ المحادثة يكلف مالًا ويُحتسب ضمن الحد الأقصى.
- **أدوات التلخيص** تتيح للوكيل تكثيف السياق المتراكم إلى ملخصات مضغوطة، مما يقلل استخدام الرموز مع الحفاظ على المعلومات الأساسية.
- **دفاتر ملاحظات الوكيل** توفر ذاكرة خارجية دائمة تبقى بعد تقليل المحادثة.

### ما الذي بنيته
- **وكيل مدرك للسياق** يحافظ على الاستمرارية عبر محادثات متعددة الأدوار
- **أداة تلخيص** (`summarize_preferences`) تسجل تفاصيل المستخدم الرئيسية بشكل مضغوط
- **محادثة متعددة الأدوار** توضح الاحتفاظ بالسياق والتعامل مع التغييرات

### تطبيقات العالم الحقيقي
- **روبوتات خدمة العملاء**: تذكر التفضيلات عبر جلسات دعم طويلة
- **المساعدون الشخصيون**: تتبع المشاريع الجارية دون الحاجة لإعادة شرح السياق
- **المدرسون التعليميون**: الحفاظ على تقدم الطلاب عبر العديد من التفاعلات

### الخطوات التالية
- تنفيذ أداة دفتر ملاحظات كاملة مع الاستمرارية المستندة إلى الملفات
- إضافة تقليص تلقائي للسجل بعد التلخيص
- الدمج مع قواعد بيانات متجهة للبحث الدلالي في الذاكرة
- بناء وكلاء يمكنهم استئناف المحادثات بعد أيام مع الاحتفاظ بالسياق الكامل


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
